# Infinite Context Memory (ICM)
## O(1) hyperbolic memory for any LLM

This notebook demonstrates ICM — a fixed-size hyperbolic state vector that replaces the KV-cache.
**260 bytes per session, regardless of conversation length.**

[![GitHub](https://img.shields.io/badge/GitHub-hyper--ssm--ultimate-blue)](https://github.com/varshinicb1/hyper-ssm-ultimate)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)]()

## Setup

Install dependencies and clone the repo.

In [ ]:
!pip install -q torch transformers sentence-transformers fastapi uvicorn
!git clone --depth 1 https://github.com/varshinicb1/hyper-ssm-ultimate.git
%cd hyper-ssm-ultimate

## Basic Usage

Create an `IcmLlm` instance and chat with infinite context memory.

In [ ]:
from hyper_ssm.llm_integration import IcmLlm

chat = IcmLlm(model_name="gpt2")
chat.create_session("demo")

reply = chat.chat("demo", "Hello! My name is Colab. Remember that.")
print("Q: Hello! My name is Colab. Remember that.")
print(f"A: {reply}")
print()

reply2 = chat.chat("demo", "What is my name?")
print("Q: What is my name?")
print(f"A: {reply2}")

## Verify O(1) Memory

The memory state stays the same size regardless of how many turns you add.

In [ ]:
session = chat._sessions["demo"]
memory = session["memory"]
print(f"Memory size: {memory.memory_size_bytes} bytes")
print(f"Utterance count: {memory._utterance_count}")
print(f"State dim: {memory.state_dim}")
print(f"\nThis is O(1) — the state will always be {memory.memory_size_bytes} bytes,")
print(f"no matter how many more turns you add.")

## Multi-turn Conversation

The LLM remembers details across many turns.

In [ ]:
questions = [
    "What's the capital of France?",
    "And what's its population?",
    "What did I ask you first?",
]

for q in questions:
    reply = chat.chat("demo", q)
    print(f"Q: {q}")
    print(f"A: {reply}")
    print()

session = chat._sessions["demo"]
memory = session["memory"]
print(f"Final memory: {memory.memory_size_bytes} bytes ({memory._utterance_count} utterances)")

## Run the Server

Start the FastAPI server in the background and test the API.

In [ ]:
import subprocess, time, json, urllib.request

# Start server in background
proc = subprocess.Popen(
    ["python", "applications/icm_server.py"],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)
time.sleep(5)

# Test health
try:
    resp = urllib.request.urlopen("http://localhost:8000/health")
    data = json.loads(resp.read())
    print("Server is running!")
    print(f"Status: {data['status']}")
    print(f"Model: {data['model']}")
except Exception as e:
    print(f"Server not ready: {e}")
finally:
    proc.terminate()

## What's Next

- **Web UI**: Run `python applications/icm_server.py` and open `http://localhost:8000/static/index.html`
- **Admin Dashboard**: Open `http://localhost:8000/admin`
- **CLI Chat**: Run `python applications/cli_chat.py`
- **GitHub**: [Star the repo!](https://github.com/varshinicb1/hyper-ssm-ultimate)

---
*Infinite Context Memory — O(1) hyperbolic memory for any LLM*